# TerraSR — Google Colab Notebook

Terrain-conditioned satellite image super-resolution.
Run cells top-to-bottom. Every pipeline stage **resumes** where it stopped, so re-running a cell after an interruption continues instead of starting over.

**Before starting:** Runtime → Change runtime type → **T4 GPU** (or better).

### Where data lives (important)
| Location | What | Why |
|---|---|---|
| **Colab local disk** (`/content/TerraSR/data`) | downloads, standardized scenes, patches, LR/HR pairs | fast for tens of thousands of small files |
| **Google Drive** (`MyDrive/TerraSR-Colab`) | checkpoints, results, WorldCover cache, **one dataset archive** | survives runtime resets |

Earlier versions of this notebook kept *all* data on Drive. Writing ~28,000 small patch files one by one through the Drive mount took hours and triggered disconnects at Patchify. Now the pipeline runs on local disk and the finished dataset is saved to Drive as a single archive (Section 6b), which later sessions restore in one copy (Section 4b).

> Local disk is wiped when Colab recycles the runtime. Anything you need to keep (checkpoints, results, the dataset archive) is written to Drive automatically.

---
| Section | What it does |
|---|---|
| 0 | GPU & environment check |
| 1 | Mount Google Drive |
| 2 | Clone repo + install deps |
| 3 | Set up data paths (local disk + Drive links) |
| 4 | Verify imports |
| **4b** | **Restore saved dataset from Drive** — returning sessions skip 5–6 |
| 5 | Download subset dataset |
| 6 | Run pipeline stages 2–6 (resumable) |
| **6b** | **Save dataset to Drive** (one archive) |
| 7 | Verify dataset |
| 8 | Single-batch smoke test |
| 9 | 5-epoch smoke training |
| 10 | Checkpoint resume test |
| 11 | Full training (all models) — resumes from Drive checkpoints |
| 12 | Evaluation → results saved to Drive |
| 13 | Download results |

**Free-tier tip:** Colab disconnects idle browser tabs and caps session length. Keep the tab open during long runs; if you get disconnected, see the resume notes in Sections 6 and 11.

## Section 0 — GPU & Environment Check

In [1]:
import subprocess, sys

# ── GPU info ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("GPU INFO")
print("=" * 60)
try:
    result = subprocess.run(["nvidia-smi",
                             "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"],
                            capture_output=True, text=True, check=True)
    name, vram, driver = [x.strip() for x in result.stdout.strip().split(",")]
    print(f"GPU      : {name}")
    print(f"VRAM     : {vram}")
    print(f"Driver   : {driver}")
except Exception as e:
    print(f"WARNING: nvidia-smi failed ({e}). Make sure a GPU runtime is selected.")

# ── PyTorch / CUDA ────────────────────────────────────────────────────────────
import torch
print()
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device   : {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM (PyTorch) : {total_gb:.1f} GB")
else:
    print()
    raise RuntimeError(
        "No CUDA GPU found. Go to Runtime → Change runtime type → T4 GPU, "
        "then reconnect and restart from Section 0."
    )

print()
print("Section 0 passed ✓")

GPU INFO
GPU      : Tesla T4
VRAM     : 15360 MiB
Driver   : 580.82.07

PyTorch  : 2.11.0+cu128
CUDA     : 12.8
GPU avail: True
Device   : Tesla T4
VRAM (PyTorch) : 15.6 GB

Section 0 passed ✓


## Section 1 — Mount Google Drive

A browser pop-up will ask you to authorise Colab to access your Drive.
After mounting, the notebook creates `MyDrive/TerraSR-Colab/` with folders for checkpoints, results, the WorldCover tile cache, and dataset archives. **The working dataset itself is not stored on Drive** — see Section 3.

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")

for subdir in ["checkpoints", "results", "logs",
               "cache/worldcover", "dataset_archives"]:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print("Drive mounted. TerraSR-Colab layout:")
print(f"  {DRIVE_ROOT}")
for p in sorted(DRIVE_ROOT.iterdir()):
    print(f"    {p.name}/")

if (DRIVE_ROOT / "data").exists():
    print("\nNote: MyDrive/TerraSR-Colab/data/ is left over from an earlier version of this")
    print("notebook. Section 5 reuses any satellite scenes already downloaded there; once your")
    print("dataset is saved (Section 6b) you can delete that folder to free Drive space.")

print()
print("Section 1 passed ✓")

Mounted at /content/drive
Drive mounted. TerraSR-Colab layout:
  /content/drive/MyDrive/TerraSR-Colab
    cache/
    checkpoints/
    data/
    dataset_archives/
    logs/
    results/

Note: MyDrive/TerraSR-Colab/data/ is left over from an earlier version of this
notebook. Section 5 reuses any satellite scenes already downloaded there; once your
dataset is saved (Section 6b) you can delete that folder to free Drive space.

Section 1 passed ✓


## Section 2 — Clone Repository & Install Dependencies

Clones the TerraSR repository from GitHub if not already present, then installs non-PyTorch dependencies from `colab/requirements-colab.txt`.  
**torch and torchvision are intentionally skipped** — Colab already provides a CUDA-enabled build.

VGG weights are pre-downloaded at the end of this section so the first training epoch does not stall on a 500 MB download.

In [3]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

# ── Clone ─────────────────────────────────────────────────────────────────────
if not REPO_DIR.exists():
    print("Cloning TerraSR repository...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/aaisha2/TerraSR.git", str(REPO_DIR)],
        check=True
    )
    print("Cloned.")
else:
    print(f"Repo already present at {REPO_DIR} — pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                   capture_output=True)
    print("Up to date.")

# ── Install deps (no torch/torchvision) ───────────────────────────────────────
req_file = REPO_DIR / "colab" / "requirements-colab.txt"
print(f"\nInstalling from {req_file.name} ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)],
    check=True
)
print("Dependencies installed.")

# ── Add repo root to sys.path ─────────────────────────────────────────────────
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

# ── Pre-warm VGG weights ───────────────────────────────────────────────────────
# Download once here so training epochs don't stall later.
print("\nPre-downloading VGG16 weights (one-time, ~528 MB)...")
import torchvision.models as tvm
_ = tvm.vgg16(weights=tvm.VGG16_Weights.IMAGENET1K_V1)
print("VGG16 weights cached.")

print()
print("Section 2 passed ✓")

Cloning TerraSR repository...
Cloned.

Installing from requirements-colab.txt ...
Dependencies installed.

Pre-downloading VGG16 weights (one-time, ~528 MB)...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 138MB/s]


VGG16 weights cached.

Section 2 passed ✓


## Section 3 — Set Up Data Paths

- `data/` is a **real folder on Colab's local disk** — all pipeline stages read and write there.
- `checkpoints/` → Drive, so training checkpoints (saved every epoch) survive runtime resets.
- `data/cache/` → Drive, so the ESA WorldCover tiles (~110 MB each) are downloaded only once.

Only a few large files ever go through the Drive mount, which Drive handles well. No config files are changed — every relative path in the research code still resolves.

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR   = Path("/content/TerraSR")
DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")
os.chdir(REPO_DIR)   # all pipeline scripts use relative paths from here

# ── 1. data/ on the LOCAL disk ─────────────────────────────────────────────────
data_dir = REPO_DIR / "data"
if data_dir.is_symlink():
    # left over from the old Drive-backed layout; removing the link does NOT
    # delete anything on Drive
    print("  removing old data/ → Drive link (your Drive files are untouched)")
    data_dir.unlink()
data_dir.mkdir(exist_ok=True)

# ── 2. durable, large-file folders stay on Drive ──────────────────────────────
# migrate WorldCover tiles cached by the old layout, so they aren't re-downloaded
old_cache = DRIVE_ROOT / "data" / "cache" / "worldcover"
new_cache = DRIVE_ROOT / "cache" / "worldcover"
new_cache.mkdir(parents=True, exist_ok=True)
if old_cache.exists():
    for tile in old_cache.glob("*.tif"):
        if not (new_cache / tile.name).exists():
            shutil.copy2(tile, new_cache / tile.name)
            print(f"  migrated WorldCover tile {tile.name}")

LINKS = {
    REPO_DIR / "checkpoints":   DRIVE_ROOT / "checkpoints",
    REPO_DIR / "data" / "cache": DRIVE_ROOT / "cache",
}
for link, target in LINKS.items():
    target.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if os.readlink(link) == str(target):
            print(f"  ✓ {link.relative_to(REPO_DIR)}/ → Drive (already linked)")
            continue
        link.unlink()
    elif link.exists():
        shutil.copytree(link, target, dirs_exist_ok=True)
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(f"  ✓ {link.relative_to(REPO_DIR)}/ → {target}")

assert not (REPO_DIR / "data").is_symlink(), "data/ must be on local disk"
assert (REPO_DIR / "checkpoints").resolve() == DRIVE_ROOT / "checkpoints"

total, used, free = shutil.disk_usage("/content")
print(f"\nLocal disk: {free/1e9:.0f} GB free of {total/1e9:.0f} GB")
print("Working directory:", os.getcwd())
print("Section 3 passed ✓")

## Section 4 — Verify Imports

Confirms every TerraSR module loads without errors.

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

errors = []

checks = [
    ("torch",                             "import torch"),
    ("rasterio",                          "import rasterio"),
    ("timm",                              "import timm"),
    ("models (registry)",                 "import models"),
    ("models.terrasr_swinir",             "from models.terrasr_swinir import build_model"),
    ("models.swinir_baseline",            "from models.swinir_baseline import build_model"),
    ("models.srcnn_baseline",             "from models.srcnn_baseline import build_model"),
    ("models.srgan_baseline",             "from models.srgan_baseline import build_model"),
    ("TerrainAwareLoss",                  "from models.losses.terrain_aware_loss import TerrainAwareLoss"),
    ("TerraSRDataset",                    "from terrasr_data import TerraSRDataset, load_terrain_index"),
    ("train_utils",                       "from training.train_utils import get_device, save_checkpoint"),
]

for label, stmt in checks:
    try:
        exec(stmt)
        print(f"  ✓ {label}")
    except Exception as e:
        print(f"  ✗ {label}: {e}")
        errors.append(label)

if errors:
    raise ImportError(f"Failed imports: {errors}. Re-run Section 2 to fix dependencies.")

print()
print("Section 4 passed ✓")

## Section 4b — Restore Saved Dataset from Drive

**Returning session?** If you already built the dataset in an earlier session (Section 6b saved it to Drive), this copies the single archive back to local disk and unpacks it — then **skip Sections 5 and 6** and continue at Section 7 (or go straight to Section 11 to resume training).

On a first run there is nothing to restore yet; the cell says so and you continue with Section 5.

In [ ]:
import shutil, tarfile, time
from pathlib import Path
import pandas as pd

REPO_DIR   = Path("/content/TerraSR")
DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")
ARCHIVE    = DRIVE_ROOT / "dataset_archives" / "terrasr_dataset.tar"
LOCAL_TRAIN = REPO_DIR / "data/dataset/train.csv"

def dataset_paths_ok():
    df = pd.read_csv(LOCAL_TRAIN)
    return len(df) > 0 and Path(df.iloc[0]["hr_path"]).exists() and Path(df.iloc[0]["lr_path"]).exists()

if LOCAL_TRAIN.exists() and dataset_paths_ok():
    print("Dataset already on local disk — nothing to restore. Continue at Section 7.")
elif ARCHIVE.exists():
    t0 = time.time()
    local_tar = Path("/content/terrasr_dataset.tar")
    print(f"Copying dataset archive from Drive ({ARCHIVE.stat().st_size/1e9:.2f} GB)...")
    shutil.copy(ARCHIVE, local_tar)
    print("Unpacking to local disk...")
    with tarfile.open(local_tar) as tf:
        try:
            tf.extractall(REPO_DIR, filter="data")
        except TypeError:          # Python without tarfile extraction filters
            tf.extractall(REPO_DIR)
    local_tar.unlink()
    assert dataset_paths_ok(), "restored manifest paths don't resolve — rebuild with Sections 5–6"
    n = len(pd.read_csv(LOCAL_TRAIN))
    print(f"Restored in {time.time()-t0:.0f}s ({n:,} training patches).")
    print("➜ Skip Sections 5 and 6 — continue at Section 7.")
else:
    print("No saved dataset on Drive yet — run Sections 5 and 6 to build it.")
    print("(Section 6b saves it to Drive, so future sessions restore it here in one step.)")

## Section 5 — Download Subset Dataset

Downloads a small, representative subset to the **local disk** to validate the end-to-end pipeline before committing to the full dataset.

**Edit the variables below to change which AOIs/events and how many files to download.**
Set `SPACENET_MAX_FILES = None` and `MAXAR_MAX_FILES = None` to download everything (whole-AOI mosaic files such as `AOI_2_Vegas_PAN_COG.tif` are skipped automatically — they duplicate the individual scenes).

Rough size guide: each SpaceNet PAN scene is 16384×16384 px ≈ 4,000 patches. The default below (3 scenes × 2 AOIs + 2 Maxar tiles) gives roughly 20,000–28,000 patches — plenty for Colab.

In [ ]:
# ── Download configuration ────────────────────────────────────────────────────
# Phase 1 (smoke test): small subset for pipeline validation
# Phase 3 (full experiment): add more AOIs/events and set max_files to None

SPACENET_AOIS       = ["AOI_2_Vegas", "AOI_5_Khartoum"]  # add more for Phase 3
SPACENET_MAX_FILES  = 3    # scenes per AOI; None = all

MAXAR_EVENTS        = ["Brazil-Flooding-May24"]            # add more for Phase 3
MAXAR_MAX_FILES     = 2    # tiles per event; None = all

print("SpaceNet AOIs :", SPACENET_AOIS, f"(max {SPACENET_MAX_FILES} files each)")
print("Maxar events  :", MAXAR_EVENTS,  f"(max {MAXAR_MAX_FILES} tiles each)")

In [ ]:
# Reuse scenes downloaded by an EARLIER version of this notebook (which stored
# them on Drive) so they aren't downloaded again. New downloads go to local disk.
import shutil
from pathlib import Path

REPO_DIR   = Path("/content/TerraSR")
DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")
OLD_RAW    = DRIVE_ROOT / "data" / "raw"

copied = 0
if OLD_RAW.exists():
    for f in OLD_RAW.rglob("*"):
        if not f.is_file() or f.suffix.lower() not in (".tif", ".tiff"):
            continue
        dest = REPO_DIR / "data/raw" / f.relative_to(OLD_RAW)
        if dest.exists() and dest.stat().st_size == f.stat().st_size:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        print(f"  copying {f.relative_to(OLD_RAW)} ({f.stat().st_size/1e6:.0f} MB) from Drive")
        shutil.copy2(f, dest)
        copied += 1
print(f"Reused {copied} scene file(s) from Drive." if copied
      else "No earlier downloads on Drive to reuse — the next cells download from AWS.")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Downloading SpaceNet PAN imagery")
print("=" * 60)

cmd = [sys.executable,
       str(REPO_DIR / "data_pipeline/01_download/download_spacenet.py"),
       "--config", str(REPO_DIR / "configs/datasets.yaml")]
for aoi in SPACENET_AOIS:
    cmd += ["--aoi", aoi]
if SPACENET_MAX_FILES is not None:
    cmd += ["--max-files", str(SPACENET_MAX_FILES)]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
if result.returncode != 0:
    raise RuntimeError("SpaceNet download failed — check error output above.")
print("\nSpaceNet download complete ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")

print("=" * 60)
print("Downloading Maxar PAN tiles")
print("=" * 60)

cmd = [sys.executable,
       str(REPO_DIR / "data_pipeline/01_download/download_maxar.py"),
       "--config", str(REPO_DIR / "configs/datasets.yaml")]
for event in MAXAR_EVENTS:
    cmd += ["--event", event]
if MAXAR_MAX_FILES is not None:
    cmd += ["--max-files", str(MAXAR_MAX_FILES)]

result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
if result.returncode != 0:
    raise RuntimeError("Maxar download failed — check error output above.")
print("\nMaxar download complete ✓")

## Section 6 — Run Data Pipeline (Stages 2–6)

Each stage is a separate cell. **Every stage resumes where it stopped:**
- standardize skips scenes already converted,
- patchify skips finished scenes and patches already on disk,
- labelling and LR/HR pair generation save progress every 500 patches and skip finished ones.

All outputs are written atomically (temp name → rename), so an interruption never leaves a half-written file that a resume would mistake for finished work.

**If you get disconnected:** reconnect. If the runtime survived (your files are still in `/content/TerraSR/data`), re-run Sections 0–4 and then the stage cell that was running — it continues. If Colab recycled the runtime, local disk is empty: re-run Sections 0–5 and this section (scenes re-download from AWS quickly).

Section 6b saves the finished dataset to Drive, so you only ever build it once.

In [ ]:
import subprocess, sys, time
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

def run_stage(label, cmd):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}", flush=True)
    t0 = time.time()
    if stream_run(cmd, cwd=REPO_DIR, label=label) != 0:
        raise RuntimeError(f"Stage failed: {label} — re-run this cell to resume")
    print(f"  {label} — done ✓ ({(time.time()-t0)/60:.1f} min)")

PY     = sys.executable
CFGS   = REPO_DIR / "configs"
DIRS   = {
    "raw":          REPO_DIR / "data/raw",
    "standardized": REPO_DIR / "data/standardized",
    "patches":      REPO_DIR / "data/patches",
    "pairs":        REPO_DIR / "data/pairs",
    "dataset":      REPO_DIR / "data/dataset",
}
print("Helpers loaded ✓")

In [ ]:
# Stage 2 — standardize scenes for each source present in data/raw/
import yaml
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text())

for source_name, src in pipeline_cfg["sources"].items():
    raw_dir = DIRS["raw"] / source_name
    if not raw_dir.exists():
        print(f"  [skip] {source_name}: not in data/raw/")
        continue
    cmd = [PY, REPO_DIR / "data_pipeline/02_standardize/standardize_scenes.py",
           "--in-dir", raw_dir,
           "--out-dir", DIRS["standardized"] / source_name,
           "--mode", src["mode"]]
    if src.get("recursive"):
        cmd.append("--recursive")
    run_stage(f"standardize/{source_name}", cmd)

In [ ]:
# Stage 3 — extract 256×256 patches
run_stage("patchify",
    [PY, REPO_DIR / "data_pipeline/03_patchify/tile_extractor.py",
     "--in-dir",  DIRS["standardized"], "--recursive",
     "--out-dir", DIRS["patches"],
     "--config",  CFGS / "patchify.yaml"])

# Stage 3 continued — filter low-content patches
run_stage("patch_filter",
    [PY, REPO_DIR / "data_pipeline/03_patchify/patch_filter.py",
     "--manifest", DIRS["patches"] / "patch_manifest.json",
     "--config",   CFGS / "patchify.yaml"])

In [ ]:
# Stage 4 — WorldCover terrain labeling
# ESA WorldCover tiles are cached in data/cache/worldcover/ on Drive after first download.
run_stage("worldcover_zonal_stats",
    [PY, REPO_DIR / "data_pipeline/04_labeling/worldcover_zonal_stats.py",
     "--manifest", DIRS["patches"] / "patch_manifest_filtered.json",
     "--config",   CFGS / "terrain_classes.yaml",
     "--only-kept"])

run_stage("assign_dominant_terrain",
    [PY, REPO_DIR / "data_pipeline/04_labeling/assign_dominant_terrain.py",
     "--manifest", DIRS["patches"] / "patch_manifest_zonal.json",
     "--config",   CFGS / "terrain_classes.yaml"])

In [ ]:
# Stage 5 — generate LR/HR GeoTIFF pairs via degradation pipeline
run_stage("make_lr_hr_pairs",
    [PY, REPO_DIR / "data_pipeline/05_degrade/make_lr_hr_pairs.py",
     "--manifest",    DIRS["patches"] / "patch_manifest_labeled.json",
     "--out-dir",     DIRS["pairs"],
     "--config",      CFGS / "degradation.yaml",
     "--only-labeled"])

In [ ]:
# Stage 6 — build unified manifest and produce train/val/test CSVs
run_stage("build_manifest",
    [PY, REPO_DIR / "data_pipeline/06_package/build_manifest.py",
     "--labeled",     DIRS["patches"] / "patch_manifest_labeled.json",
     "--degradation", DIRS["pairs"]   / "degradation_manifest.json",
     "--out-dir",     DIRS["dataset"]])

run_stage("split_train_val_test",
    [PY, REPO_DIR / "data_pipeline/06_package/split_train_val_test.py",
     "--manifest", DIRS["dataset"] / "dataset_manifest.parquet",
     "--config",   CFGS / "split.yaml"])

run_stage("dataset_stats",
    [PY, REPO_DIR / "data_pipeline/06_package/dataset_stats.py",
     "--manifest", DIRS["dataset"] / "dataset_manifest_split.parquet",
     "--config",   CFGS / "split.yaml"])

print("\nAll pipeline stages complete ✓")

## Section 6b — Save Dataset to Drive

Packs the training-ready dataset (`data/pairs/` + `data/dataset/`) into **one** archive and copies it to `MyDrive/TerraSR-Colab/dataset_archives/terrasr_dataset.tar`. One large file transfers quickly through the Drive mount, unlike thousands of small ones.

Future sessions restore it with Section 4b in a single step instead of re-running the pipeline. Re-run this cell whenever you rebuild or extend the dataset.

In [ ]:
import os, shutil, tarfile, time
from pathlib import Path

REPO_DIR   = Path("/content/TerraSR")
DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")
ARCHIVE    = DRIVE_ROOT / "dataset_archives" / "terrasr_dataset.tar"
MEMBERS    = ["data/pairs", "data/dataset"]

assert (REPO_DIR / "data/dataset/train.csv").exists(), "run Section 6 first"

def skip_temp(tarinfo):
    return None if ".partial" in tarinfo.name else tarinfo

t0 = time.time()
local_tar = Path("/content/terrasr_dataset.tar")
print("Building archive on local disk...")
with tarfile.open(local_tar, "w") as tf:   # uncompressed: GeoTIFFs are already compressed
    for m in MEMBERS:
        tf.add(REPO_DIR / m, arcname=m, filter=skip_temp)
size_gb = local_tar.stat().st_size / 1e9
print(f"  {size_gb:.2f} GB")

print("Copying to Drive...")
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
tmp = ARCHIVE.with_suffix(".tar.partial")
shutil.copy(local_tar, tmp)
os.replace(tmp, ARCHIVE)      # appears on Drive only once fully copied
local_tar.unlink()

print(f"\nSaved {ARCHIVE} ({size_gb:.2f} GB) in {(time.time()-t0)/60:.1f} min")
print("Section 6b passed ✓ — next session, Section 4b restores this in one step.")

## Section 7 — Verify Dataset

Confirms that the stage 6 manifests are readable, the terrain distribution looks sensible, and one batch loads correctly through the PyTorch Dataset.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

REPO_DIR = Path("/content/TerraSR")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from terrasr_data import TerraSRDataset, load_terrain_index

terrain_index = load_terrain_index(REPO_DIR / "configs/terrain_classes.yaml")

for split_name, csv_path in [("train", "data/dataset/train.csv"),
                              ("val",   "data/dataset/val.csv"),
                              ("test",  "data/dataset/test.csv")]:
    p = REPO_DIR / csv_path
    if not p.exists():
        print(f"  {split_name}: {csv_path} not found — check stage 6 output")
        continue
    df = pd.read_csv(p)
    print(f"\n{split_name} ({len(df):,} patches):")
    if "terrain_label" in df.columns:
        print(df["terrain_label"].value_counts().to_string())

# Load one batch
print("\n--- Batch load test ---")
ds = TerraSRDataset(REPO_DIR / "data/dataset/train.csv",
                    terrain_index=terrain_index,
                    normalize="per_patch_max", augment=False)
loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
lr, hr, terrain_idx, meta = next(iter(loader))
print(f"LR  shape : {tuple(lr.shape)}  dtype: {lr.dtype}  range: [{lr.min():.3f}, {lr.max():.3f}]")
print(f"HR  shape : {tuple(hr.shape)}  dtype: {hr.dtype}  range: [{hr.min():.3f}, {hr.max():.3f}]")
print(f"terrain_idx: {terrain_idx.tolist()}")
print(f"terrain_label: {meta['terrain_label']}")

print()
print("Section 7 passed ✓")

## Section 8 — Single-Batch Smoke Test

Builds TerraSR and runs one forward + backward pass without a full training loop. This confirms the GPU, model, and loss all work together before committing to any training time.

In [ ]:
import sys, yaml
from pathlib import Path
import torch

REPO_DIR = Path("/content/TerraSR")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import models
from terrasr_data import load_terrain_index
from models.losses.terrain_aware_loss import TerrainAwareLoss
from training.train_utils import get_device

device = get_device()
print(f"Device: {device}")
torch.cuda.reset_peak_memory_stats()

# Build model
cfg = yaml.safe_load((REPO_DIR / "configs/train_terrasr.yaml").read_text())
model = models.build("terrasr", cfg["model"]).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"TerraSR params: {n_params/1e6:.2f}M")

# Build loss (perceptual disabled to avoid another VGG forward during smoke test)
terrain_index = load_terrain_index(REPO_DIR / "configs/terrain_classes.yaml")
loss_cfg = yaml.safe_load((REPO_DIR / "configs/terrain_aware_loss.yaml").read_text())
for w in loss_cfg["per_terrain"].values():
    w["perceptual"] = 0.0
loss_cfg["default"]["perceptual"] = 0.0
criterion = TerrainAwareLoss(loss_cfg, terrain_index).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

# Synthetic batch (no dataset needed for the smoke test)
BATCH = 2
LR_SIZE, HR_SIZE = 64, 128   # 2× scale factor
lr_t = torch.rand(BATCH, 1, LR_SIZE, LR_SIZE, device=device)
hr_t = torch.rand(BATCH, 1, HR_SIZE, HR_SIZE, device=device)
terrain_idx = torch.tensor([0, 2], device=device)   # Urban, Forest

# Forward + backward
optimizer.zero_grad()
sr = model(lr_t, terrain_idx)
loss, components = criterion(sr, hr_t, terrain_idx)
loss.backward()
optimizer.step()

mem_mb = torch.cuda.max_memory_allocated() / 1e6
print(f"SR output shape : {tuple(sr.shape)}")
print(f"Loss            : {loss.item():.4f}")
print(f"Components      : {components}")
print(f"Peak GPU memory : {mem_mb:.0f} MB")

print()
print("Section 8 passed ✓")

## Section 9 — Smoke Training (5 Epochs)

Runs a short training run with conservative settings to confirm the full loop works — data loading, loss computation, checkpointing to Drive — before committing to the overnight run.

**Checkpoint will appear at:** `MyDrive/TerraSR-Colab/checkpoints/smoke_terrasr/last.pth`

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

print("=" * 60)
print("Smoke training — TerraSR, 5 epochs")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_terrasr.py"),
    "--config", str(REPO_DIR / "configs/train_terrasr.yaml"),
    "--override",
        "train.epochs=5",
        "train.batch_size=4",
        "train.num_workers=2",
        "train.out_dir=checkpoints/smoke_terrasr",
        "loss.disable_perceptual=true",   # skip VGG for speed
]
stream_run(cmd, cwd=REPO_DIR, label="smoke training", check=True)

# Verify checkpoint exists on Drive
ckpt = REPO_DIR / "checkpoints/smoke_terrasr/last.pth"
assert ckpt.exists(), f"Expected checkpoint at {ckpt}"
size_mb = ckpt.stat().st_size / 1e6
print(f"\nCheckpoint saved: {ckpt} ({size_mb:.1f} MB)")
print("Section 9 passed ✓")

## Section 10 — Checkpoint Resume Test

Simulates a runtime reset by re-running the same training command (no `--fresh`).  
The startup banner reports what was already trained; because all 5 smoke epochs are done it should report `already trained all 5 epochs ... nothing to do`. That is the crash-recovery path working — with fewer epochs completed it would instead print `RESUMING` and the epoch it continues from.


In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

print("=" * 60)
print("Resume test — the banner below reports what was already trained")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_terrasr.py"),
    "--config", str(REPO_DIR / "configs/train_terrasr.yaml"),
    "--override",
        "train.epochs=5",           # already done; should print 'nothing to do'
        "train.batch_size=4",
        "train.num_workers=2",
        "train.out_dir=checkpoints/smoke_terrasr",
        "loss.disable_perceptual=true",
]
stream_run(cmd, cwd=REPO_DIR, label="resume test", check=True)

print()
print("Section 10 passed ✓")
print("('already trained all 5 epochs ... nothing to do' is the expected result —")
print(" it means the resume logic found the completed checkpoint and skipped it.)")

## Section 11 — Full Training

Trains all four models (SRCNN, SRGAN, SwinIR, TerraSR) at full config. This is the overnight run.

**Batch size guidance:**
- T4 (15 GB) → `batch_size=8`
- L4 (22 GB) → `batch_size=12`
- A100 (40 GB) → `batch_size=16` (original research setting)

### Watching progress

Each training cell now streams its output live. You should see, within a minute of starting:

```
==============================================================
model      : srcnn  (0.06M params)
device     : cuda
data       : 24160 train / 3020 val patches, batch 8 -> 3020 batches/epoch
out_dir    : checkpoints/srcnn
RESUMING   : 12/100 epochs already done (12%) - continuing at epoch 13
best so far: val PSNR 31.84 dB
to train   : 88 more epoch(s)
==============================================================
[02:14:07] epoch 13/100 started (3020 batches)
[02:15:11]   epoch 13/100  batch 50/3020 (  2%)  loss 0.0231  0.78 it/s  epoch ETA 63m22s
[02:31:40] epoch  13/100 done  loss 0.0198  val PSNR 31.91 dB  best 31.91 dB  <- best, best.pth updated  (17m33s)  |  87 epoch(s) left, ETA 25h27m
```

So at any moment you can see which epoch is running, how far into it you are, and the ETA for the epoch and for the run.

Every completed epoch is also appended to **`checkpoints/<model>/training_log.csv`** on Drive. That file survives everything the notebook output does not — a dropped browser tab, a cleared output, a recycled runtime — so it is the reliable record of how far training got. **Section 11b** reads it back.

**If the runtime terminates:** re-run Sections 0–4, then **Section 4b** (restores the dataset from Drive to local disk), then Section 11 — each model resumes from its last completed epoch automatically, and finished models are skipped.


In [ ]:
import torch

# Detect available VRAM and suggest a batch size
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU VRAM: {total_gb:.1f} GB")

if total_gb >= 38:
    BATCH_SIZE = 16
elif total_gb >= 20:
    BATCH_SIZE = 12
else:
    BATCH_SIZE = 8

print(f"Auto-selected batch_size: {BATCH_SIZE}")
print("(Edit BATCH_SIZE below to override if you get OOM errors.)")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

print("=" * 60)
print("Training SRCNN baseline")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_baseline.py"),
    "--config", str(REPO_DIR / "configs/train_baseline.yaml"),
    "--override",
        "model.name=srcnn",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        "train.out_dir=checkpoints/srcnn",
]

stream_run(cmd, cwd=REPO_DIR, label="SRCNN training", check=True)
print("SRCNN done ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

print("=" * 60)
print("Training SRGAN baseline")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_baseline.py"),
    "--config", str(REPO_DIR / "configs/train_baseline.yaml"),
    "--override",
        "model.name=srgan",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        "train.out_dir=checkpoints/srgan",
]

stream_run(cmd, cwd=REPO_DIR, label="SRGAN training", check=True)
print("SRGAN done ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

print("=" * 60)
print("Training SwinIR baseline")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_baseline.py"),
    "--config", str(REPO_DIR / "configs/train_baseline.yaml"),
    "--override",
        "model.name=swinir",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        "train.out_dir=checkpoints/swinir",
]

stream_run(cmd, cwd=REPO_DIR, label="SwinIR training", check=True)
print("SwinIR done ✓")

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
# stream_run prints the child process output line by line as it happens.
# subprocess.run() buffers it until the process exits, which hides hours
# of training progress and loses it entirely if the session is cut.
from colab_utils import stream_run

print("=" * 60)
print("Training TerraSR (terrain-conditioned model)")
print("=" * 60)

cmd = [
    sys.executable, str(REPO_DIR / "training/train_terrasr.py"),
    "--config", str(REPO_DIR / "configs/train_terrasr.yaml"),
    "--override",
        f"train.batch_size={BATCH_SIZE}",
        "train.num_workers=2",
        # leave loss.disable_perceptual unset (false) — use full terrain-aware loss
]

stream_run(cmd, cwd=REPO_DIR, label="TerraSR training", check=True)
print("TerraSR done ✓")

## Section 11b — Training Progress / Status

How far did training actually get? This reads the checkpoints and epoch logs on Drive — no GPU, no model building — and reports per model: epochs completed, where the next run will resume, best validation PSNR, and the most recent epoch lines.

Run it:
- after reconnecting, to see what survived
- before Section 11, to see what is left to train
- before Section 12, to check every model finished

(It can also be run from a second Colab notebook pointed at the same Drive folder while training is still going.)


In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/TerraSR")
sys.path.insert(0, str(REPO_DIR / "colab"))
from colab_utils import stream_run

TAIL = 8   # how many recent epochs to list per model

stream_run([sys.executable, REPO_DIR / "training/training_status.py",
            "--dir", REPO_DIR / "checkpoints", "--tail", TAIL],
           cwd=REPO_DIR, label="status")


## Section 12 — Evaluation

Runs the full evaluation suite against all four trained models using the held-out test split:
- Global PSNR/SSIM table (with bicubic baseline)
- Per-terrain PSNR breakdown
- Results report (Markdown + `.docx`)

The report is written to **`MyDrive/TerraSR-Colab/results/`**, so it survives runtime resets.

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_DIR   = Path("/content/TerraSR")
DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")

CKPTS = [
    str(REPO_DIR / "checkpoints/srcnn/best.pth"),
    str(REPO_DIR / "checkpoints/srgan/best.pth"),
    str(REPO_DIR / "checkpoints/swinir/best.pth"),
    str(REPO_DIR / "checkpoints/terrasr/best.pth"),
]

# Check all checkpoints exist
missing = [c for c in CKPTS if not Path(c).exists()]
if missing:
    print("WARNING: some checkpoints not found — evaluation will run on available ones:")
    for m in missing:
        print(f"  MISSING: {m}")
    CKPTS = [c for c in CKPTS if Path(c).exists()]

TEST_CSV   = str(REPO_DIR / "data/dataset/test.csv")
SPLIT_MF   = str(REPO_DIR / "data/dataset/dataset_manifest_split.csv")
RESULTS    = DRIVE_ROOT / "results" / "results_report"   # on Drive — survives resets

sys.path.insert(0, str(REPO_DIR / "colab"))
from colab_utils import stream_run   # live output instead of a silent wait

def run_eval(label, cmd):
    print(f"\n--- {label} ---")
    if stream_run(cmd, cwd=REPO_DIR, label=label) != 0:
        print(f"  WARNING: {label} did not complete")

PY = sys.executable

run_eval("PSNR/SSIM table",
    [PY, REPO_DIR / "evaluation/eval_psnr_ssim.py",
     "--test-csv", TEST_CSV, "--with-bicubic",
     "--checkpoints", *CKPTS])

run_eval("Per-terrain PSNR",
    [PY, REPO_DIR / "evaluation/eval_per_terrain.py",
     "--test-csv", TEST_CSV,
     "--checkpoints", *CKPTS])

run_eval("Results report",
    [PY, REPO_DIR / "evaluation/make_results_report.py",
     "--test-csv",       TEST_CSV,
     "--split-manifest", SPLIT_MF,
     "--out",            str(RESULTS),
     "--checkpoints",    *CKPTS])

print()
print("Section 12 complete ✓")
print(f"Results saved to: {RESULTS.parent}  (MyDrive/TerraSR-Colab/results/)")

## Section 13 — Download Results

Downloads the results report directly to your local machine. The files also remain on Google Drive.

In [ ]:
from google.colab import files
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/TerraSR-Colab")
RESULTS    = DRIVE_ROOT / "results"

to_download = sorted(RESULTS.glob("results_report*"))

if not to_download:
    print(f"No result files found in {RESULTS}. Run Section 12 first.")
else:
    print(f"Downloading {len(to_download)} result file(s)...")
    for f in to_download:
        print(f"  {f.name}")
        files.download(str(f))
    print()
    print("All results downloaded to your local machine ✓")
    print("They are also available on Drive at:")
    print("  MyDrive/TerraSR-Colab/results/")